# 16장. 미세 조정할까, 프롬프트로 할까

같은 과제(영화평 감정 분류)를 세 가지 방법으로 푼다. 나이브 베이즈, Kanana에 프롬프트, Qwen3-0.6B에 LoRA 미세 조정이다. 고르는 일은 모두 개발 자료로 하고, 남겨 둔 시험 자료는 마지막에 한 번만 연다. 정답률과 함께 학습 시간, 문장 하나의 지연, 메모리를 재서 언제 어느 쪽이 나은지 계산한다.

**준비:** `pip install "transformers<5" huggingface_hub pandas`. 9장에서 받은 Qwen3-0.6B와 Kanana 1.5 2.1B, 1장의 NSMC `ratings_train.txt`, `ratings_test.txt`(CC0 1.0)가 있어야 한다. 저자의 맥(CPU 스레드 2개)에서 끝까지 실행하는 데 27분쯤 걸렸다.

## 16.1 자료를 나누고 시험 몫은 남겨 두기

In [ ]:
import os, re, json, math, time, sys, random, copy, platform, warnings, collections, statistics
import numpy as np
import torch
import torch.nn.functional as F
os.environ["HF_HUB_DISABLE_XET"] = "1"
warnings.filterwarnings("ignore")
torch.set_num_threads(2)
import transformers
from transformers import AutoTokenizer, AutoModelForCausalLM
from transformers.utils import logging as hf_logging
hf_logging.set_verbosity_error()

PATH = {"Qwen3-0.6B": "data/models/Qwen__Qwen3-0.6B", "Kanana-1.5-2.1B": "data/models/kakaocorp__kanana-1.5-2.1b-instruct-2505"}
tok = {n: AutoTokenizer.from_pretrained(p) for n, p in PATH.items()}
load = lambda n: AutoModelForCausalLM.from_pretrained(PATH[n], dtype=torch.float32).eval()
def chat_ids(n, text):
    extra = {"enable_thinking": False} if n.startswith("Qwen3") else {}
    return tok[n].apply_chat_template([{"role": "user", "content": text}], tokenize=True, add_generation_prompt=True, **extra)
print(sys.version.split()[0], torch.__version__, transformers.__version__, platform.machine(), torch.get_num_threads())

In [ ]:
def read(name):
    rows = [line.split("\t") for line in open(name, encoding="utf-8").read().split("\n")[1:] if line.count("\t") == 2]
    return [(text.strip(), int(label)) for _, text, label in rows if 5 <= len(text.strip()) <= 80]
pick = random.Random(0)
train_all, test_all = read("ratings_train.txt"), read("ratings_test.txt")
pick.shuffle(train_all); pick.shuffle(test_all)
balance = lambda rows, n: [r for r in rows if r[1] == 1][:n // 2] + [r for r in rows if r[1] == 0][:n // 2]
POOL = train_all[:20000]                                  # 학습에 쓸 수 있는 몫(레이블이 있다고 본다)
DEV = balance(train_all[20000:24000], 300)                # 고르는 데만 쓰는 개발 자료
BIG = train_all[24000:124000]                             # 레이블이 많을 때를 보려고 따로 둔 10만 개
TEST = balance(test_all, 600)                             # 남겨 둔 시험 자료: 맨 끝에 한 번만 연다
L400, L2000 = balance(POOL[:4000], 400), balance(POOL, 2000)
pick.shuffle(L400); pick.shuffle(L2000)
seen = {t for t, _ in POOL} | {t for t, _ in BIG}
print(f"학습 몫 {len(POOL):,}개(400개·2,000개를 여기서 뽑는다), 개발 {len(DEV)}개, 큰 학습 몫 {len(BIG):,}개, 시험 {len(TEST)}개")
print(f"개발 문장 가운데 학습 몫에도 있는 문장 {sum(t in seen for t, _ in DEV)}개, 시험 문장 가운데 {sum(t in seen for t, _ in TEST)}개")

## 16.2 세 가지 방법을 개발 자료로 재기

In [ ]:
# ① 나이브 베이즈(12장의 판정기와 같다): 어절과 글자 두 개씩을 특징으로 센다
feats = lambda s: set(s.split()) | {s[i:i + 2] for i in range(len(s) - 1)}
class NB:
    def __init__(self, rows, alpha=1.0):
        self.c, self.d, self.a = [collections.Counter(), collections.Counter()], [0, 0], alpha
        for text, y in rows:
            self.c[y].update(feats(text)); self.d[y] += 1
        self.V = len(set(self.c[0]) | set(self.c[1])); self.T = [sum(c.values()) for c in self.c]
    def score(self, text):
        lp = [math.log(self.d[y] / sum(self.d)) + sum(math.log((self.c[y][f] + self.a) / (self.T[y] + self.a * self.V)) for f in feats(text)) for y in (0, 1)]
        return lp[1] - lp[0]
acc = lambda f, rows: float(np.mean([(f(t) > 0) == (y == 1) for t, y in rows]))
DEVR, KEEP, COST = {}, {}, {}
for name, rows in [("400", L400), ("2,000", L2000), ("10만", BIG)]:
    for a in [0.1, 1.0]:
        lap = time.perf_counter(); m = NB(rows, a); sec = time.perf_counter() - lap
        DEVR[f"나이브 베이즈 {name}개 α={a}"] = acc(m.score, DEV); KEEP[f"나이브 베이즈 {name}개 α={a}"] = m; COST[f"나이브 베이즈 {name}개 α={a}"] = {"train": sec, "labels": len(rows)}
        print(f"나이브 베이즈 레이블 {name:>5}개 α={a}: 학습 {sec:5.1f}초, 개발 정답률 {DEVR[f'나이브 베이즈 {name}개 α={a}']:.1%}", flush=True)

In [ ]:
# ② 프롬프트(15장과 같은 꼴): 답 칸 첫 토큰에서 '긍정'과 '부정'의 점수를 견준다. 앞부분 KV 캐시를 다시 쓴다
TASK = "영화평을 읽고 감정을 '긍정'이나 '부정'으로 답하세요."
LAB = {1: "긍정", 0: "부정"}
def body(shots, x):
    return "".join(f"영화평: {d}\n감정: {LAB[l]}\n\n" for d, l in shots) + f"영화평: {x}\n감정:"
def draw(k, seed):
    cand = [r for r in POOL if 10 <= len(r[0]) <= 60]
    return random.Random(seed).sample(cand, k) if k else []
def score(model, n, texts, batch=16):
    """프롬프트 글 목록마다 (긍정 점수 − 부정 점수). 모두 같은 앞부분을 한 번만 계산한다(15장)."""
    full = [chat_ids(n, t) for t in texts]
    p = 0
    while all(len(f) > p + 1 and f[p] == full[0][p] for f in full):
        p += 1
    pre, suf = full[0][:p], [f[p:] for f in full]
    ia, ib = tok[n].encode("긍정", add_special_tokens=False)[0], tok[n].encode("부정", add_special_tokens=False)[0]
    pad = tok[n].pad_token_id if tok[n].pad_token_id is not None else 0
    with torch.inference_mode():
        base = model(input_ids=torch.tensor([pre]), use_cache=True).past_key_values
    out = []
    for k in range(0, len(suf), batch):
        ss = suf[k:k + batch]; L = max(map(len, ss)); B = len(ss)
        cache = copy.deepcopy(base); cache.batch_repeat_interleave(B)
        ids = torch.tensor([s + [pad] * (L - len(s)) for s in ss])
        mask = torch.tensor([[1] * (p + len(s)) + [0] * (L - len(s)) for s in ss])
        with torch.inference_mode():
            lg = model(input_ids=ids, attention_mask=mask, past_key_values=cache).logits
        z = lg[torch.arange(B), torch.tensor([len(s) - 1 for s in ss])]
        out += (z[:, ia] - z[:, ib]).tolist()
    return out
PROMPTS = {"Kanana 예시 0개": ("Kanana-1.5-2.1B", 0, 0), "Kanana 예시 4개(뽑기 0)": ("Kanana-1.5-2.1B", 4, 0),
           "Kanana 예시 4개(뽑기 1)": ("Kanana-1.5-2.1B", 4, 1), "Kanana 예시 4개(뽑기 2)": ("Kanana-1.5-2.1B", 4, 2),
           "Qwen3 예시 0개": ("Qwen3-0.6B", 0, 0)}
prompt_text = lambda k, s, x: TASK + "\n\n" + body(draw(k, s), x)
for n in ["Kanana-1.5-2.1B", "Qwen3-0.6B"]:
    model = load(n)
    for name, (mn, k, s) in PROMPTS.items():
        if mn != n:
            continue
        lap = time.perf_counter()
        d = score(model, n, [prompt_text(k, s, x) for x, _ in DEV])
        DEVR[name] = float(np.mean([(v > 0) == (y == 1) for v, (_, y) in zip(d, DEV)]))
        COST[name] = {"train": 0.0, "labels": k, "dev_sec": time.perf_counter() - lap}
        print(f"{name:<22} 개발 정답률 {DEVR[name]:.1%}, '긍정'이라고 한 비율 {np.mean([v > 0 for v in d]):.1%} ({time.perf_counter() - lap:.0f}초)", flush=True)
    del model

In [ ]:
# ③ 미세 조정: Qwen3-0.6B 모든 층의 어텐션 투영 행렬 넷(q·k·v·o)에 LoRA(r=8)를 붙여 답에만 손실을 준다(10·11장과 같다)
N = "Qwen3-0.6B"; t = tok[N]; END = t.convert_tokens_to_ids("<|im_end|>")
ASK = "다음 영화평이 긍정이면 '긍정', 부정이면 '부정'이라고만 답하세요.\n영화평: {}"
class LoRALinear(torch.nn.Module):
    def __init__(self, base, r, alpha):
        super().__init__()
        self.base, self.scale = base, alpha / r
        self.A = torch.nn.Parameter(torch.empty(r, base.in_features)); self.B = torch.nn.Parameter(torch.zeros(base.out_features, r))
        torch.nn.init.kaiming_uniform_(self.A, a=math.sqrt(5))
    def forward(self, x):
        return self.base(x) + (x @ self.A.T) @ self.B.T * self.scale
def add_lora(model, r=8):
    for p_ in model.parameters():
        p_.requires_grad_(False)
    for mod in list(model.modules()):
        for name, sub in list(mod.named_children()):
            if name in ("q_proj", "k_proj", "v_proj", "o_proj") and isinstance(sub, torch.nn.Linear):
                setattr(mod, name, LoRALinear(sub, r, 2 * r))
    return model
def example(text, y):
    p_ = chat_ids(N, ASK.format(text)); a = t.encode(LAB[y], add_special_tokens=False) + [END]
    return p_ + a, [-100] * len(p_) + a
def lora_dev(model, rows):
    model.eval()
    d = score(model, N, [ASK.format(x) for x, _ in rows])
    return float(np.mean([(v > 0) == (y == 1) for v, (_, y) in zip(d, rows)]))
STATES = {}
for name, rows, epochs in [("400개", L400, 3), ("2,000개", L2000, 1)]:
    torch.manual_seed(0); model = add_lora(load(N))
    opt = torch.optim.AdamW([p_ for p_ in model.parameters() if p_.requires_grad], lr=1e-4, weight_decay=0.0)
    data = [example(x, y) for x, y in rows]; spent = 0.0
    for ep in range(1, epochs + 1):
        model.train(); order = random.Random(ep).sample(range(len(data)), len(data)); lap = time.perf_counter(); tot = 0.0
        for k in range(0, len(order), 8):
            rr = [data[i] for i in order[k:k + 8]]; L = max(len(r[0]) for r in rr)
            ids = torch.tensor([r[0] + [t.pad_token_id] * (L - len(r[0])) for r in rr])
            lab = torch.tensor([r[1] + [-100] * (L - len(r[1])) for r in rr])
            mask = torch.tensor([[1] * len(r[0]) + [0] * (L - len(r[0])) for r in rr])
            loss = model(input_ids=ids, attention_mask=mask, labels=lab).loss
            opt.zero_grad(); loss.backward(); opt.step(); tot += loss.item() * len(rr)
        spent += time.perf_counter() - lap
        key = f"LoRA {name} {ep}에폭"
        STATES[key] = {n_: (m_.A.detach().clone(), m_.B.detach().clone()) for n_, m_ in model.named_modules() if isinstance(m_, LoRALinear)}
        DEVR[key] = lora_dev(model, DEV); COST[key] = {"train": spent, "labels": len(rows)}
        print(f"{key:<16} 학습 손실 {tot / len(data):.4f}, 누적 학습 {spent:5.0f}초, 개발 정답률 {DEVR[key]:.1%}", flush=True)
    del model

## 16.3 고르기, 그리고 비용 재기

In [ ]:
FAM = {"나이브 베이즈": [k for k in DEVR if k.startswith("나이브")], "프롬프트": [k for k in DEVR if k in PROMPTS], "미세 조정": [k for k in DEVR if k.startswith("LoRA")]}
CHOSEN = {f: max(ks, key=lambda k: (DEVR[k], -COST[k]["train"])) for f, ks in FAM.items()}     # 개발 정답률이 같으면 학습이 짧은 쪽
for f, ks in FAM.items():
    print(f"{f}: " + ", ".join(f"{k.replace('나이브 베이즈 ', '').replace('LoRA ', '')} {DEVR[k]:.1%}" for k in ks))
print("개발 자료로 고른 것:", CHOSEN)

In [ ]:
# 문장 하나를 배치 1로 처리하는 시간(개발 문장 앞 30개의 중앙값)과 모델의 메모리(32비트 가중치)
def merged_lora(key):
    model = add_lora(load(N))
    for n_, m_ in model.named_modules():
        if isinstance(m_, LoRALinear):
            m_.A.data, m_.B.data = STATES[key][n_][0].clone(), STATES[key][n_][1].clone()
    for mod in list(model.modules()):                                  # LoRA를 가중치에 합친다(11장)
        for name, sub in list(mod.named_children()):
            if isinstance(sub, LoRALinear):
                with torch.no_grad():
                    sub.base.weight += (sub.B @ sub.A) * sub.scale
                setattr(mod, name, sub.base)
    return model.eval()
def timed(fn, items):
    ts = []
    for x in items:
        lap = time.perf_counter(); fn(x); ts.append(time.perf_counter() - lap)
    return statistics.median(ts)
SAMPLE = [x for x, _ in DEV[:30]]
LAT, MEM = {}, {}
m = KEEP[CHOSEN["나이브 베이즈"]]
LAT["나이브 베이즈"] = timed(m.score, SAMPLE); MEM["나이브 베이즈"] = sum(len(c) for c in m.c) * 2 * 8 / 1e9   # 두 클래스 표의 항목마다 16바이트로 어림(키와 파이썬 자료 구조의 부담은 뺀 하한)
mn, k, s = PROMPTS[CHOSEN["프롬프트"]]
model = load(mn); MEM["프롬프트"] = sum(p_.numel() for p_ in model.parameters()) * 4 / 1e9
LAT["프롬프트"] = timed(lambda x: score(model, mn, [prompt_text(k, s, x)]), SAMPLE)        # 배치 1, 앞부분도 매번 계산
del model
model = merged_lora(CHOSEN["미세 조정"]); MEM["미세 조정"] = sum(p_.numel() for p_ in model.parameters()) * 4 / 1e9
LAT["미세 조정"] = timed(lambda x: score(model, N, [ASK.format(x)]), SAMPLE)
del model
for f in FAM:
    print(f"{f:<7} ({CHOSEN[f]}): 문장 하나 {LAT[f] * 1000:8.2f}ms, 메모리 {MEM[f]:.3f}GB, 학습 {COST[CHOSEN[f]]['train']:6.1f}초, 레이블 {COST[CHOSEN[f]]['labels']:,}개")

## 16.4 남겨 둔 시험 자료로 한 번만 평가하기

In [ ]:
# 개발 자료로 고른 세 가지만, 시험 자료 600개로 한 번 잰다
FINAL = {}
FINAL["나이브 베이즈"] = acc(KEEP[CHOSEN["나이브 베이즈"]].score, TEST)
mn, k, s = PROMPTS[CHOSEN["프롬프트"]]
model = load(mn)
d = score(model, mn, [prompt_text(k, s, x) for x, _ in TEST]); FINAL["프롬프트"] = float(np.mean([(v > 0) == (y == 1) for v, (_, y) in zip(d, TEST)]))
del model
model = merged_lora(CHOSEN["미세 조정"]); FINAL["미세 조정"] = lora_dev(model, TEST); del model
se = lambda p_: math.sqrt(p_ * (1 - p_) / len(TEST))
for f in FAM:
    print(f"{f:<7} 개발 {DEVR[CHOSEN[f]]:.1%} → 시험 {FINAL[f]:.1%} (±{1.96 * se(FINAL[f]):.1%}, 95% 구간의 반폭)")

In [ ]:
# 해서는 안 되는 일을 일부러 해 본다: 후보를 모두 시험 자료로 재고, 시험 점수가 가장 높은 것을 고르면?
ALLT = {}
for key, m in KEEP.items():
    ALLT[key] = acc(m.score, TEST)
for key in STATES:
    model = merged_lora(key); ALLT[key] = lora_dev(model, TEST); del model
best_t = max(ALLT, key=ALLT.get)
print("미세 조정·나이브 베이즈 후보 10개의 시험 정답률:", {k: round(v, 4) for k, v in ALLT.items()})
print(f"시험으로 고르면 {best_t} {ALLT[best_t]:.1%}, 개발로 고른 것은 미세 조정 {FINAL['미세 조정']:.1%}·나이브 베이즈 {FINAL['나이브 베이즈']:.1%}")
gap = {k: ALLT[k] - DEVR[k] for k in ALLT}
print("시험 − 개발(%포인트):", {k: round(v * 100, 1) for k, v in gap.items()})

In [ ]:
import glob
import matplotlib.pyplot as plt
from matplotlib import font_manager
for fp in (glob.glob(os.path.expanduser("~/Library/Fonts/NotoSansCJK*.otf")) + glob.glob("/usr/share/fonts/opentype/noto/NotoSansCJK*.ttc")):
    font_manager.fontManager.addfont(fp)
plt.rcParams.update({"font.family": ["Noto Sans CJK KR", "DejaVu Sans"], "font.size": 8, "axes.unicode_minus": True})
BLUE, ORANGE, GREEN, PURPLE, RED, GRAY = "#2F6FDB", "#F28C28", "#1AA37A", "#8A5CD1", "#E5484D", "#9AA3AF"
COLF = {"나이브 베이즈": GRAY, "프롬프트": GREEN, "미세 조정": BLUE}
fig, axes = plt.subplots(1, 2, figsize=(5.2, 2.6), dpi=260)
a = axes[0]
fams = list(FAM)
a.bar(np.arange(3) - 0.18, [DEVR[CHOSEN[f]] * 100 for f in fams], 0.36, color=[COLF[f] for f in fams], alpha=0.45, label="개발")
a.bar(np.arange(3) + 0.18, [FINAL[f] * 100 for f in fams], 0.36, color=[COLF[f] for f in fams], label="시험(한 번)")
a.errorbar(np.arange(3) + 0.18, [FINAL[f] * 100 for f in fams], yerr=[1.96 * se(FINAL[f]) * 100 for f in fams], fmt="none", color="#1f2937", lw=0.9, capsize=2)
a.set_xticks(np.arange(3), ["나이브\n베이즈", "프롬프트\n(Kanana)", "미세 조정\n(Qwen3 LoRA)"]); a.set_ylim(50, 100)
a.set_title("① 정답률(%): 연한 막대 개발, 진한 막대 시험", loc="left", fontsize=8.2, color="#374151")
a = axes[1]
n_ = np.logspace(0, 6, 200)
for f in fams:
    a.plot(n_, (COST[CHOSEN[f]]["train"] + n_ * LAT[f]) / 3600, color=COLF[f], lw=1.4, label=f)
a.set_xscale("log"); a.set_yscale("log"); a.set_xlabel("분류할 문장 수"); a.set_ylabel("학습 + 처리 시간(단위: 시간)")
a.set_title("② 쓰는 양에 따른 전체 시간", loc="left", fontsize=8.2, color="#374151")
a.legend(frameon=False, fontsize=8, loc="upper left")
for a in axes:
    a.spines[["top", "right"]].set_visible(False)
fig.tight_layout(); fig.savefig("fig_16_decide.png"); plt.close(fig)